## Using hyperbolic library

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import hyperbolic as hb

In [ ]:
import math
import numpy as np

import drawSvg as draw
from drawSvg import Drawing
import hyperbolic
from hyperbolic import euclid, util
from hyperbolic.euclid.shapes import Circle as ECircle
from hyperbolic.poincare.shapes import *
from hyperbolic.poincare import Transform
from hyperbolic.poincare.util import radialEuclidToPoincare, radialPoincareToEuclid, \
                                     poincareToEuclidFactor, triangleSideForAngles
import hyperbolic.tiles as htiles

class Circle(Circle):
    def __init__(self, projShape, center=None, r=None):
        super().__init__(projShape)
        if not isinstance(projShape, ECircle):
            raise ValueError('projShape must be a euclidean circle')
        self.projShape = projShape
        if r is None or center is None:
            de0 = math.hypot(projShape.cx, projShape.cy)
            de1 = de0 - projShape.r
            de2 = de0 + projShape.r
            dh1 = radialEuclidToPoincare(de1)
            dh2 = radialEuclidToPoincare(de2)
        if r is None:
            r = (dh2 - dh1) / 2
        if center is None:
            cr = (dh2 + dh1) / 2
            theta = math.atan2(projShape.cx, projShape.cy)
            center = Point.fromHPolar(cr, theta)
        self.r = r
        self.center = center

In [ ]:
def drawTiles(drawing, tiles):
    for tile in tiles:
        global t
        t = tile
        d.draw(tile, hwidth=0.02, fill='white')
    for tile in tiles:
        d.draw(tile, drawVerts=True, hradius=0.15, hwidth=0.02,
                     fill='black', opacity=0.6)

In [ ]:
class GraphTileLayout(htiles.TileLayout):
    def tilePlane(self, startTile, depth=2, return_edges=False):
        edges = set()
        tiles = [startTile]
        boundary = [(side, startTile) for side in startTile.sides]
        for j in range(depth):
            boundary2 = []
            i = 0
            while i < len(boundary):
                tile = self.placeTile(boundary[i][0])
                tiles.append(tile)
                edges.add((tile, boundary[i][1]))
                sides = tile.permutedSides()
                o = 1
                p = len(sides)
                if i == 0:
                    if sides[o] == boundary[-1][0]:
                        o += 1
                        edges.add((tile, boundary[-1][1]))
                        boundary.pop()
                else:
                    if sides[o] == boundary2[-1][0]:
                        o += 1
                        edges.add((tile, boundary2[-1][1]))
                        boundary2.pop()
                    if sides[p-1] == boundary[(i + 1) % len(boundary)][0]:
                        p -= 1
                        edges.add((tile, boundary[(i + 1) % len(boundary)][1]))
                        i += 1
                    if sides[p-1] == boundary2[0][0]:
                        p -= 1
                        edges.add((tile, boundary2[0][1]))
                        boundary2.pop(0)
                boundary2.extend([[side, tile] for side in sides[o:p]])
                i += 1
            boundary = boundary2
        return (tiles, edges) if return_edges else tiles

In [ ]:
from itertools import chain
#def construct_circumcircle(polygon):
def get_circumcenter(polygon):
    if isinstance(polygon, hyperbolic.tiles.Tile):
        polygon = polygon.toPolygon()
    
    c = ECircle.fromPoints(*chain(*[[v.x, v.y] for v in polygon.vertices[:3]]))
    #return c.cx, c.cy
    c = Circle(c)
    #print(c.center.x, c.center.y, c.r)
    return c.center.x, c.center.y
#return vs[0]
#v = construct_circumcircle(tiles[5])
#type(vs[0].__dict__)

In [ ]:
import eucare as ec
import networkx as nx

plotting_kwargs = {
    'figsize': (5, 5),
    'render_faces': False,
    'render_vertices': False,
    'render_edges': True,
    'face_inset': 0,
    'line_width': 3,
}
render_settings = plotting_kwargs

In [ ]:
# Control the orientation that tiles are placed together
class TileLayoutIsosceles(GraphTileLayout):
    def calcGenIndex(self, code):
        ''' Controls which type of tile to place '''
        return 0
    def calcTileTouchSide(self, code, genIndex):
        ''' Controls tile orientation '''
        try:
            side, colors = code
            return 2 - side
        except TypeError:
            return 0
    def calcSideCodes(self, code, genIndex, touchSide, defaultCodes):
        ''' Controls tile side codes '''
        try:
            side, colors = code
            # 0=red, 1=orange, 2=yellow, 3=lime, 4=green, 5=blue, 6=pink
            if side != 1:
                if side == 0: shift = -1
                elif side == 2: shift = 1
                else: shift = 0
                nc = len(colors)
                newColors = [colors[(i+shift)%nc] for i in range(nc)]
            else:
                newColors = [colors[0], colors[1], colors[6], colors[4],
                    colors[3], colors[5], colors[2]]
        except TypeError:
            nc = q1
            newColors = [(i+code)%nc for i in range(nc)]
        return [(side, newColors) for side in range(3)]
    
q1 = 7  # Number of polygons around some points
q2 = 6  # Number of polygons around other points
depth = 14  # How far from the center to draw tiles

# Calculate isosceles triangle
assert q2 > 4 and q2 % 2 == 0, 'q2 must be even and at least 6'
phi1, phi2 = math.pi*2/q1, math.pi*2/q2
# Side lengths
s0 = triangleSideForAngles(phi1, phi2, phi2)
s1 = triangleSideForAngles(phi2, phi2, phi1)
s2 = s0
pt0 = Point.fromHPolar(0,0)
pt1 = Point.fromHPolar(s0,0)
pt2 = Point.fromHPolar(s2,phi1)
# Circumcircle
circumcirc = euclid.shapes.Circle.fromPoints(*pt0, *pt1, *pt2)
r = radialEuclidToPoincare(circumcirc.r)
ptCenter = Point.fromEuclid(circumcirc.cx, circumcirc.cy)
# Translate triangle to center
transCenter = Transform.shiftOrigin(ptCenter, pt0)
ptc0, ptc1, ptc2 = transCenter(pt0, pt1, pt2)
centerPoints = (ptc0, ptc1, ptc2)
tile = htiles.Tile(centerPoints)


# Calculate weave width
# For right triangle: tan(A) = tanh(opp) / sinh(adj)
# => opp = atanh(tan(A) * sinh(adj))
rInsc = math.atanh(math.tan(phi2/2) * math.sinh(s1/2))  # Inscribed circle radius
h = math.atanh(math.tan(phi2) * math.sinh(s1/2))  # Triangle height
centerDiff = r - (h - rInsc)

tGen = htiles.TileGen.fromCenterTile(tile)

decoratorLate = htiles.TileDecoratorLateInit()

tLayout = TileLayoutIsosceles()
tLayout.addGenerator(tGen, (0,)*4, decoratorLate)
startTile = tLayout.startTile(code=2,rotateDeg=0,centerCorner=False)

tiles, edges = tLayout.tilePlane(startTile, depth=depth, return_edges=True)

d = draw.Drawing(2, 2, origin='center')
d.draw(euclid.shapes.Circle(0, 0, 1), fill='#ddd')
drawTiles(d, tiles)

d.setRenderSize(w=400)
d.saveSvg('isosceles-{}-{}.svg'.format(q1, q2))
d

In [ ]:
# Regular tesselation
p = 7
q = 3
depth = 5

theta = math.pi*2/q
phi = math.pi*2/p
r = triangleSideForAngles(theta/2, phi, theta/2)
print(r)

tGen = htiles.TileGen.makeRegular(p, hr=r, skip=1)

tLayout = GraphTileLayout()
tLayout.addGenerator(tGen, (0,)*p)

startTile = tLayout.defaultStartTile(rotateDeg=90)

tiles, edges = tLayout.tilePlane(startTile, depth=depth, return_edges=True)

d = Drawing(2, 2, origin='center')
d.draw(euclid.shapes.Circle(0, 0, 1), fill='silver')
drawTiles(d, tiles)

d.setRenderSize(w=400)
d.saveSvg('tileTriangleSquare.svg')
d

In [ ]:
G = nx.Graph()
circumcenters = {tile: get_circumcenter(tile) for tile in tiles}
#print(circumcenters.values())

G.add_edges_from([(circumcenters[t1], circumcenters[t2])
                 for t1, t2 in edges])

plt.figure()
nx.draw(G, {n: np.array(n) for n in G.nodes})#, list(G.nodes))
plt.gca().set_aspect('equal')
plt.show()
# assert False
G = ec.conversions.EHEG_from_nx(G)
G.show(**plotting_kwargs)

ps = G.get_position_view(return_vertices=False)
zs = np.array([complex(*vals) for vals in ps])

#zs = np.arctanh(zs)

#directions = np.exp(1j * np.linspace(0, 2*np.pi, 3, endpoint=False))
#zs = np.sum([d.conj()*np.log(1-d*zs) for d in directions], axis=0)

ps[:] = np.stack([zs.real, zs.imag], axis=-1)


#G = ec.conway.join_graph()(G)

G.show(**plotting_kwargs)

In [ ]:

from eucare.base import signed_area

for e in G.halfedges:
    if THIS_WAY in e:
        del e[THIS_WAY]

def pointing_away(e):
    return (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()

# for normal
# eps = 1e-8
# for e in G.halfedges:
#     radial_difference = (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()
#     if radial_difference > eps or (abs(radial_difference) < eps and signed_area(np.array([[0, 0], e.orig['pos'], e.dest['pos']])) < 0):
#         e[THIS_WAY] = True
#         assert THIS_WAY not in e.rev
        
        
vertices = list(G.vertices)
central_vertex = vertices[np.argmin([np.linalg.norm(v['pos']) for v in vertices])]
hops_to_central = {central_vertex: 0}
boundary = {central_vertex}
i = 1
while boundary:
    next_boundary = set()
    for v in boundary:
        for v2 in v.vertex_iter():
            if v2 not in hops_to_central and v2:
                next_boundary.add(v2)
                hops_to_central[v2] = i
    i += 1
    boundary = next_boundary
    
eps = 1e-8
for e in G.halfedges:
    radial_difference = (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()
    if radial_difference > eps or (abs(radial_difference) < eps and signed_area(np.array([[0, 0], e.orig['pos'], e.dest['pos']])) < 0):
        if hops_to_central[e.orig] % 2 == 0:
            e = e.rev
        e[THIS_WAY] = True
        assert THIS_WAY not in e.rev

## wrong
# def squared_edge_length(e):
#     return ((e.orig['pos'] - e.dest['pos'])**2).sum()

# verts = G.vertices
# verts = sorted(verts, key=lambda v: -max(*[squared_edge_length(e) for e in v.outgoing_iter()]))
# for v in verts:
#     for e in v.outgoing_iter():
#         e[THIS_WAY] = True
#         if THIS_WAY in e.rev:
#             del e.rev[THIS_WAY]

# for join
# for f in G.faces:
#     if f.order() == 4:
#         for e in f.halfedge_iter():
#             if e.orig.order() == 7:
#                 e[THIS_WAY] = True                

In [ ]:
from eucare.overlap import THIS_WAY, assign_shrink_rotate_creases

def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

SRG = ec.reciprocal_figures.shrink_rotate_graph(G)
SRG.recompute_lengths_and_angles()
assign_shrink_rotate_creases(SRG)

colors = {
    0: (0, 0, 0),
    1: (1, 0, 0),
    -1: (0, 0, 1)
}
for e in SRG.halfedges:
    e['color_key'] = colors[e.attributes.get('crease_assignment', 0)]
# join unneccessary boundary vertices
to_join = []
for v in SRG.vertices:
    if v.on_border() and v.order() == 2:
        to_join.append(v)
for v in to_join:
    SRG.join_vertex(v)
SRG.recompute_lengths_and_angles()

SRG.show(**plotting_kwargs)

mks = max_kawasaki_sum(SRG)
print(mks)

In [ ]:
%matplotlib notebook
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection
from eucare.plotting import set_equal_aspect
from eucare.reciprocal_figures import random_directed_set

def reshrinkrotate(alpha, factor, global_scale=1):
    for f in faces:
        if 'rotation_center' not in f.attributes:
            continue
        ps, vs = np.array([[v['base_pos'], v] for v in f.vertex_iter()]).T
        ps = np.stack(ps)
        rotation_center = f['rotation_center']

        ps = rotation_center + (ps - rotation_center) @ ec.base.rotation_matrix(alpha) * factor
        
        if global_scale != 1:
            ps *= global_scale
            
        for v, p in zip(vs, ps):
            v['pos'] = p
            
def get_segments(edges):
    return np.array([[e.orig['pos'], e.dest['pos']] for  e in edges])

def get_polys(faces):
    return [[v['pos'] for v in f.vertex_iter()] for f in faces]

edges = list(random_directed_set(SRG.halfedges))
faces = list(SRG.faces)

#%timeit SRG.show(**render_settings)
#%timeit reshrinkrotate(np.pi/9, 0.7)
#%timeit get_segments(edges)
#%timeit get_polys(faces)

segments = get_segments(edges)
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1, 1, 1)
lc = LineCollection(segments, antialiased=True, color='k', linewidth=1)

pc = PolyCollection(get_polys(faces), antialiased=True, color='k')
pc.set_alpha(0.1)

polys = ax.add_collection(pc)
lines = ax.add_collection(lc)

ax.autoscale()
set_equal_aspect()
plt.draw()

alpha_slider = widgets.FloatSlider(0.166666, min=-1, max=1, step=0.02)
factor_slider = widgets.FloatSlider(0.58, min=0, max=6, step=0.05)

last_reparametrized = False
def update(alpha, factor, folded=False, reparametrized=False, scale_folded=False, show_lines=False, show_polys=True):
    alpha = alpha * np.pi
    global last_reparametrized
    if not last_reparametrized:
        gamma = factor / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1)
        beta = np.arccos(np.sin(alpha) / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1))
    else:
        gamma = factor
        beta = alpha
        # TODO: sign
        alpha = np.arccos((gamma + np.sin(beta)) / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1))
        factor = gamma / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1)
    
    if reparametrized is not last_reparametrized:
        # adjust slider values
        last_reparametrized = reparametrized
        if reparametrized:
            alpha_slider.value = beta / np.pi
            factor_slider.value = gamma
        else:
            alpha_slider.value = alpha / np.pi
            factor_slider.value = factor
    
    if not folded:
        reshrinkrotate(alpha, factor)
    else:
        factor_folded = gamma / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1)
        alpha_folded = np.sign(alpha) * np.arccos((gamma - np.sin(beta)) / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1))
        reshrinkrotate(alpha_folded, factor_folded, 
                       global_scale=1 if not scale_folded else factor/factor_folded)
    lines.set_segments(get_segments(edges) if show_lines else [])
    polys.set_paths(get_polys(faces) if show_polys else [])
    #ax.draw_artist(lc)
    fig.canvas.draw_idle()
    #print(f'gamma {gamma}, beta {beta * 360 / (2 * np.pi)}')

widgets.interact(
    update,
    alpha=alpha_slider,
    factor=factor_slider,
    
);

In [ ]:
from scipy.optimize import minimize_scalar, basinhopping
from copy import copy

def angle_to_height(G, angle):
    border_positions = np.array([v['pos'] for v in G.border_vertex_iter()])
    rot_border_positions = border_positions @ np.array([[np.cos(angle)], [-np.sin(angle)]])
    return np.max(rot_border_positions) - np.min(rot_border_positions)

def optimize_rotation(G):
    border_positions = np.array([v['pos'] for v in G.border_vertex_iter()])

    def angle_to_height(angle):
#         if isinstance(angle, np.ndarray):
#             angle = angle[0]
        rot_border_positions = border_positions @ np.array([[np.cos(angle)], [-np.sin(angle)]])
        return np.max(rot_border_positions) - np.min(rot_border_positions)
    
    #minimizer_kwargs = dict(method="L-BFGS-B", bounds=[[0, 2*np.pi]])
    #result = basinhopping(angle_to_height, x0=0, stepsize=np.pi/100, minimizer_kwargs=minimizer_kwargs)
    #angle = result['x'][0]
    
    angles = np.linspace(0, np.pi, 10000)
    heights = [angle_to_height(a) for a in angles]
    plt.figure()
    plt.plot(angles, heights)
    plt.show()
    angle = angles[np.argmin(heights)]
    print(angle)
    ps = G.get_position_view(return_vertices=False)
    ps[:] = ps @ np.array([[np.cos(angle), np.sin(angle)], [-np.sin(angle), np.cos(angle)]])
    
def min_edge_length(G):
    edges = copy(G.halfedges)
    min_length = np.inf
    while edges:
        e = edges.pop()
        edges.remove(e.rev)
        min_length = min(((e.orig['pos'] - e.dest['pos'])**2).sum(), min_length)
    return np.sqrt(min_length)

min_foldable_length = 0.5 #cm
min_edge_length(SRG)

optimize_rotation(SRG)
sheet_height = angle_to_height(SRG, 0) * min_foldable_length / min_edge_length(SRG)
print('sheet heigth:', sheet_height, 'cm')

# angles = np.linspace(0, np.pi/2, 100)
# heights = [angle_to_height(SRG, a) for a in angles]
# angles[np.argmin(heights)]
# plt.figure()
# plt.plot(angles, heights)
# plt.show();

In [ ]:
from eucare.overlap import fold_complete
SRG.recompute_lengths_and_angles()
result = fold_complete(SRG.copy(), overlap_eps=1e-7, area_eps=0)
render_settings['render_faces'] = False
result['CP'].show(**render_settings)
result['folded_view_top'].show(**render_settings)
result['folded_view_bottom'].show(**render_settings)

In [ ]:
import os
from eucare.redering import SvgwriteRenderer
from eucare.overlap import save_results

path = 'nice_images/hyperbolic_7-3_huge_f1.5'
save_results(result, path, render_settings)
plotter = SvgwriteRenderer()
plotter.render_graph(os.path.join(path, 'cp_for_cutting.svg'), result['CP'], height=sheet_height)

In [ ]:
p1 = 5
p2 = 4
q = 2

theta1, theta2 = math.pi*2/p1, math.pi*2/p2
phiSum = math.pi*2/q
r1 = triangleSideForAngles(theta1/2, phiSum/2, theta2/2)
r2 = triangleSideForAngles(theta2/2, phiSum/2, theta1/2)

tGen1 = htiles.TileGen.makeRegular(p1, hr=r1, skip=1)
tGen2 = htiles.TileGen.makeRegular(p2, hr=r2, skip=1)

tLayout = htiles.TileLayout()
tLayout.addGenerator(tGen1, (1,)*p1)
tLayout.addGenerator(tGen2, (0,)*p2)
startTile = tLayout.defaultStartTile(rotateDeg=90)

tiles = tLayout.tilePlane(startTile, depth=2)

d = Drawing(2, 2, origin='center')
d.draw(euclid.shapes.Circle(0, 0, 1), fill='silver')
drawTiles(d, tiles)

d.setRenderSize(w=400)
d.saveSvg('tileTriangleSquare.svg')
d

In [ ]:
t.__dict__

In [ ]:
type(e.p1)

## Using eucare only

### HyperbolicHEG

In [ ]:
import eucare as ec
from eucare.half import EuclideanPositionHEG
from eucare.base import unit_vector, angle_to_axis, edge_lengths_and_in_angles, signed_area

class HyperbolicHEG(EuclideanPositionHEG):
    @staticmethod
    def construct_next_point(a, b, angle, length):
        """construct the point c such that angle(a, b, c)=angle and |bc|=length"""
        a, b = real_to_complex(np.array([a, b]))
        offset = b
        a, b = complex_to_real(hyperbolic_translation(-offset, np.array([a, b])))
        c = EuclideanPositionHEG.construct_next_point(a, b, angle, np.tanh(length / 2))
        return complex_to_real(hyperbolic_translation(offset, real_to_complex(c)))
        

In [ ]:
import eucare as ec
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
from eucare.example_tilesets import align_tiles


# class RegularPolygonalPrototile(ec.prototiles.PolygonalProtoTile):
#     """base class for regular polygonal prototiles in hyperbolic, eucildean or spherical geometry"""
#     def __init__(self, n_sides, tiles_per_vertex, edge_length=None, **super_kwargs):
        
#         alpha = np.pi / n_sides
#         beta = np.pi / tiles_per_vertex

#         r = np.sqrt(1 / ((np.cos(beta) / np.sin(alpha))**2 - 1))
#         print('r', r)
#         c = np.sqrt(1 + 2 * r**2 - 2 * r * np.sqrt(1 + r**2) * np.cos(np.pi/2 - alpha - beta))
#         print('c', c)
#         l = 4*np.arctanh(abs(hyperbolic_translation(r - np.sqrt(1+r**2), real_to_complex(c*unit_vector(alpha)))))
#         pritn('l', l)
        
#         in_angles = [2 * beta] * n_sides
#         edge_length = l #if edge_length is None else edge_length
#         edge_lengths = [edge_length] * n_sides
        
#         self.points = np.tanh(r/2) * unit_vector(np.linspace(0, 2*np.pi, n_sides, endpoint=False))
#         super().__init__(in_angles, edge_lengths, **super_kwargs)
        
        
class RegularHyperbolicPrototile(ec.prototiles.PolygonalProtoTile):
    """base class for regular polygonal prototiles in hyperbolic, eucildean or spherical geometry"""
    def __init__(self, n_sides, tiles_per_vertex, **super_kwargs):
        
        alpha = np.pi / n_sides
        beta = np.pi / tiles_per_vertex

        r = np.sqrt(1 / ((np.cos(beta) / np.sin(alpha))**2 - 1))
        c = np.sqrt(1 + 2 * r**2 - 2 * r * np.sqrt(1 + r**2) * np.cos(np.pi/2 - alpha - beta))
        l = 4 * np.arctanh(np.linalg.norm(hyperbolic_translation(r - np.sqrt(1+r**2), real_to_complex(c*unit_vector(alpha)))))
        
        in_angles = [2 * beta] * n_sides
        edge_lengths = [l] * n_sides
        
        self.points = c * unit_vector(np.linspace(0, 2*np.pi, n_sides, endpoint=False))
        super().__init__(in_angles=in_angles, edge_lengths=edge_lengths, **super_kwargs)
        
def curved_platonic(n, m):
    t = RegularHyperbolicPrototile(n, m, edge_labels=['a'] * n)
    align_tiles(t, 'a', t, 'a')
    return [t]

In [ ]:
from copy import copy
from eucare.prototiles import ProtoTile
from eucare.half import *

# this is incorrect! but the concept is good, could make this member functions of the graph classes
def vertexBFS(G, v0):
    visited = set()
    boundary = {v0}
    while boundary:
        v = boundary.pop()
        visited.add(v)
        for v_new in v.vertex_iter():
            if v_new in visited or v_new in boundary:
                continue
            boundary.add(v_new)
            yield v
            
            
def from_tiles(tiles, rings=2, vertex_based=True, base_tile=-1, add_positions=True, hyperbolic=False):
    if isinstance(base_tile, int):
        base_tile = tiles[base_tile]
    if isinstance(base_tile, ProtoTile):
        base_tile = base_tile.make_graph(add_positions=add_positions)[0]
    if add_positions:
        if not hyperbolic:
            G = EuclideanPositionHEG(other=base_tile)
        else:
            G = HyperbolicHEG(other=base_tile)
    else:
        G = InAngleHEG(other=base_tile)
    if vertex_based:
        for i in range(rings):
            for v in [h.orig for h in G.border_edges()]:
                while v in G.vertices and v.on_border():
                    G.execute_edge_instruction(v.get_outgoing_border())
    else:
        for i in range(rings):
            for h in G.border_edges():
                if h.on_border() and h in G.halfedges:
                    if 'instruction' in h.attributes:
                        G.execute_edge_instruction(h)
    return G


G = from_tiles(curved_platonic(5,4), rings=3, add_positions=True, vertex_based=True, hyperbolic=True)
print('plotting..')
#print(G.order, [v.order() for v in G.vertices], [f.order() for f in G.faces])

render_settings = dict(
    width=2000,
    height=2000,
    figsize=(8, 8),
    #scale=100,
    render_edges=True,
    render_faces=True,
    render_vertices=False,
    face_inset=0.01,
    for_cutting = False,
    line_width=7
)
G.show(**render_settings)
G = ec.conway.kis_graph()(G)
G = ec.conway.dual_graph()(G)
G.show(**render_settings)

In [ ]:
unit_vector(0)

In [ ]:
alpha = np.pi / 7
beta = np.pi / 3

r = np.sqrt(1 / ((np.cos(beta) / np.sin(alpha))**2 - 1))
print('r', r)
c = np.sqrt(1 + 2 * r**2 - 2 * r * np.sqrt(1 + r**2) * np.cos(np.pi/2 - alpha - beta))
print('c', c)
l = 4*np.arctanh(abs(hyperbolic_translation(r - np.sqrt(1+r**2), real_to_complex(c*unit_vector(alpha)))))
l
#np.tanh(c/2)

In [ ]:
# how to compute positions in certain model?
# 1. choose some edge to point upwards from the origin. boundary = {e0}
# 2. for e in boundary: go around star, add e.nex to new boundary
# repeat 2. until done
# need 'transform edge onto edge' operation (given edges have same length)
# could be obtained just from translation to / from origin + rotation around it

### barycentric coordinates
from https://www.math.auckland.ac.nz/deptdb/dept_reports/498.pdf

In [ ]:
from eucare.base import *

#def angle(a, b, c):
#    return (angle_to_axis(a - b) - angle_to_axis(c - b)) % (2*np.pi)


def barycentric_to_poincare(a, b, c, x, y):
    # b is the origin of the barycentric coordinate system
    a, b, c = c, a, b 
    a, b = hyperbolic_translation(-c, np.array([a, b]))
    # fixme!!
    area = hyperbolic_triangle_area(0, a, b)
    expy = np.exp(-1j * y * area)
    expx = np.exp(1j * x * area)
    z = (a * (1 - expy) - b * (1 - expx)) / (a.conjugate() * b * expx - a * b.conjugate() * expy)
    return hyperbolic_translation(c, z)

def hyperbolic_translation(a, z):
    """see https://en.wikipedia.org/wiki/Gyrovector_space#Poincar%C3%A9_disc/ball_model_and_M%C3%B6bius_addition"""
    return (a + z) / (1 + a.conjugate() * z)

def is_critical(z):
    return np.allclose(np.linalg.norm(z), np.ones_like(z))

def complex_to_real(z):
    return np.stack([z.real, z.imag], axis=-1)

def real_to_complex(x):
    assert x.shape[-1] == 2
    return x[..., 0] + 1j * x[..., 1]

def hyperbolic_angle(a, b, c):
    # move b to origin
    if is_critical(b): # this makes this impossible 
        return 0
    a, b, c = complex_to_real(hyperbolic_translation(-b, np.array([a, b, c])))
    return angle(a, b, c)

def hyperbolic_triangle_area(a, b, c):
    return np.pi - hyperbolic_angle(a, b, c) - hyperbolic_angle(b, c, a) - hyperbolic_angle(c, a, b)

In [ ]:
def disk_figure(**figure_kwargs):
    plt.figure(**figure_kwargs)
    plt.gca().set_aspect('equal')
    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.axis('off')
    circle = plt.Circle((0, 0), 1, color='k', linewidth=0, alpha=0.1)
    plt.gca().add_artist(circle)

In [ ]:
def poincare_to_hyperboloid(z):
    pts = complex_to_real(z)
    squared_norm = (pts**2).sum(-1, keepdims=True)
    return np.concatenate([(1 + squared_norm), 2*pts], axis=-1) / (1 - squared_norm)

def hyperboloid_to_poincare(v):
    return real_to_complex(v[..., 1:] / (1 + v[..., :1]))

def hyperboloid_centroid(vs, ms=None, axis=None):
    # see https://projecteuclid.org/download/pdf_1/euclid.cmp/1104252873
    if axis is None:
        assert len(vs.shape) == 2
        axis = 0
    ms = np.ones(list(vs.shape[:-1]) + [1], dtype=vs.dtype) if ms is None else ms
    mean = (vs * ms[..., None]).mean(axis)
    mean /= np.sqrt(mean[..., :1]**2 - (mean[..., 1:]**2).sum(-1, keepdims=True))
    return mean

def poincare_centroid(zs, ms=None, axis=None):
    return hyperboloid_to_poincare(hyperboloid_centroid(poincare_to_hyperboloid(zs), ms, axis))

def trianglecoords_to_poincare(a, b, c, x, y):
    extra_dims = len(x.shape) if isinstance(x, np.ndarray) else 0
    return poincare_centroid(np.array([a, b, c])[(slice(None),) + (None,)*(extra_dims)], 
                             ms=np.stack([x, y, 1-x-y]), 
                             axis=-2-extra_dims)

In [ ]:
poincare_centroid(np.array([0.1, 0, 0.1j]), np.array([1, 1, 1]))

In [ ]:
# plt.figure(figsize=(12,12))
# circle = plt.Circle((0, 0), 1, color='k', linewidth=0, alpha=0.1)
# plt.gca().add_artist(circle)

def plot_triangle(a, b, c, coord_func, offset=None):
    if hyperbolic_triangle_area(a, b, c) < 0:
        a, b, c = c, b, a
    steps = 1000
    if offset:
        a, b, c = hyperbolic_translation(offset, np.array([a, b, c]))
    for t in np.linspace(0, 1, 15):
        pts = coord_func(a, b, c, np.linspace(0, t, steps), np.linspace(t, 0, steps))
        pts = pts if not offset else hyperbolic_translation(-offset, pts)
        plt.plot(pts.real, pts.imag)
        pts = coord_func(a, b, c, np.linspace(0, t, steps),np.linspace(1-t, 1-t, steps))
        pts = pts if not offset else hyperbolic_translation(-offset, pts)
        plt.plot(pts.real, pts.imag)
        pts = coord_func(a, b, c, np.linspace(1-t, 1-t, steps), np.linspace(t, 0, steps))
        pts = pts if not offset else hyperbolic_translation(-offset, pts)
        plt.plot(pts.real, pts.imag)
    #center = coord_func(a, b, c, 1/3, 1/3)
    #plt.scatter(*complex_to_real(np.array([a, b, c, center])).T)
    
a = 0.7
b = 0.3 + 0.8j
c = -0.6 - 0.6j
d = -0.7 + 0.5j

disk_figure(figsize=(10, 10))
plot_triangle(a, b, c, barycentric_to_poincare)
plot_triangle(d, c, b, barycentric_to_poincare)
# plot_triangle(d, c, b, barycentric_to_poincare, offset=0.5)  # check independence to choice of origin
plt.title('barycentric coordinates')

disk_figure(figsize=(10, 10))
plot_triangle(a, b, c, trianglecoords_to_poincare)
plot_triangle(d, c, b, trianglecoords_to_poincare)
# plot_triangle(d, c, b, trianglecoords_to_poincare, offset=0.5)  # check independence to choice of origin
plt.title('center of mass coordinates')
plt.show();

In [ ]:
np.tanh(10)

In [ ]:
2*np.arctanh(0.999)

In [ ]:
def apply_mobius(mat, points):
    return (mat[0, 0] * points + mat[0, 1]) / (mat[1, 0] * points + mat[1, 1])

class MobiusTransform():
    def __init__(self, mat):
        if not isinstance(mat, np.ndarray):
            mat = np.array(mat)
        assert mat.shape == (2, 2), f'{mat.shape}'
        self.mat = mat
        
    def __call__(self, points):
        return apply_mobius(self.mat, points)
    
    def __matmul__(self, other):
        assert isinstance(other, MobiusTransform), f'{type(other)}'
        return MobiusTransform(self.mat @ other.mat)
    
    def __pow__(self, exponent):
        return MobiusTransform(np.linalg.matrix_power(self.mat, exponent))
    
    def __repr__(self):
        return f'MobiusTransform({self.mat.tolist()})'

def argument(z):
    return np.arctan2(z.imag, z.real)

def translation(z1, z2=None):
    if z2 is None:
        return MobiusTransform([[1, z1], [z1.conjugate(), 1]])
    m1 = translation(-z2)
    m2 = translation(-m1(z1))
    m3 = translation(z2)
    return m3 @ m2 @ m1

def rotation(alpha, z=None):
    if z is None:
        return MobiusTransform([[np.exp(1j*phi), 0], [0, 1]])
    return origin_translation(z) @ rotation(alpha) @ origin_translation(-z)

def sq_abs(z):
    return z.real**2 + z.imag**2

def distance(a, b):
    return np.arccosh(1 + (2 * sq_abs(a-b)) / (1 - sq_abs(a)) / (1 - sq_abs(b)))

def angle(a, b, c):
    return ec.base.angle

In [ ]:
a = 0.2 - 0.3j
b = 0.2 - 0.5j
c = 0.8 - 0.4j
d = -0.8 - 0.4j
f = translation(a, b)
n = 100
corners = [a, b, c, d, a]
pts = np.concatenate([np.linspace(z1, z2, n) for z1, z2 in zip(corners[:-1], corners[1:])])
disk_figure()
for i in range(-10, 10):
    plt.plot(*complex_to_real((f**i)(pts)).T)
plt.show();

## Geometry Objects

TODO: think about if and if yes how I want to vecorize everything

In [ ]:
import numpy as np
from eucare.base import Geometry

class EuclideanGeometry(Geometry):
    @classmethod
    def origin(cls):
        return np.array([0, 0])

    @classmethod
    def translation(cls, p1, p2):
            
        def translate(p):
            return p + p2 - p1
        
        return translate

    @classmethod
    def _rotate_around_origin(cls, a1):
        rot_mat = np.array([[np.cos(a1), np.sin(a1)], [-np.sin(a1), np.cos(a1)]])
        
        def origin_rotate(p):
            return p @ rot_mat
        
        return origin_rotate

    @classmethod
    def center_of_mass(cls, points, masses=None):
        assert len(points.shape) == 2 and points.shape[-1] == 2, f'{points.shape}'
        if masses is not None:
            masses = masses / np.sum(masses) * len(points)
            points = points * masses[..., None]
        return np.mean(points, axis=0)

    @classmethod
    def distance_to_origin(cls, p):
        return np.linalg.norm(p)

    @classmethod
    def angle_to_axis(cls, p):
        return np.arctan2(p[..., 1], p[..., 0])

    @classmethod
    def point_along_axis(cls, x):
        return np.array([x, 0])

In [ ]:
def apply_mobius(mat, points):
    return (mat[0, 0] * points + mat[0, 1]) / (mat[1, 0] * points + mat[1, 1])


class MobiusTransform():
    def __init__(self, mat):
        if not isinstance(mat, np.ndarray):
            mat = np.array(mat)
        assert mat.shape == (2, 2), f'{mat.shape}'
        self.mat = mat
        
    def __call__(self, points):
        return apply_mobius(self.mat, points)
    
    def __matmul__(self, other):
        assert isinstance(other, MobiusTransform), f'{type(other)}'
        return MobiusTransform(self.mat @ other.mat)
    
    def __pow__(self, exponent):
        return MobiusTransform(np.linalg.matrix_power(self.mat, exponent))
    
    def __repr__(self):
        return f'MobiusTransform({self.mat.tolist()})'
    
    
# TODO: maybe have another class for the hyperboloid model

def complex_to_real(z):
    return np.stack([z.real, z.imag], axis=-1)

def real_to_complex(x):
    assert x.shape[-1] == 2
    return x[..., 0] + 1j * x[..., 1]

def poincare_to_hyperboloid(z):
    pts = complex_to_real(z)
    squared_norm = (pts**2).sum(-1, keepdims=True)
    return np.concatenate([(1 + squared_norm), 2*pts], axis=-1) / (1 - squared_norm)

def hyperboloid_to_poincare(v):
    return real_to_complex(v[..., 1:] / (1 + v[..., :1]))

def hyperboloid_centroid(vs, ms=None, axis=None):
    # see https://projecteuclid.org/download/pdf_1/euclid.cmp/1104252873
    if axis is None:
        assert len(vs.shape) == 2
        axis = 0
    ms = np.ones(list(vs.shape[:-1]) + [1], dtype=vs.dtype) if ms is None else ms
    mean = (vs * ms[..., None]).mean(axis)
    mean /= np.sqrt(mean[..., :1]**2 - (mean[..., 1:]**2).sum(-1, keepdims=True))
    return mean

def poincare_centroid(zs, ms=None, axis=None):
    return hyperboloid_to_poincare(hyperboloid_centroid(poincare_to_hyperboloid(zs), ms, axis))


class PoincareDiskModel(Geometry):
    @classmethod
    def origin(cls):
        return 0 + 0j

    @classmethod
    def translation(cls, p1, p2):
        if p2 == 0:
            p1, p2 = 0, -p1
        if p1 == 0:
            return MobiusTransform([[1, p2], [p2.conjugate(), 1]])
        m1 = cls.translation(p2, 0)
        m2 = cls.translation(0, -m1(p1))
        m3 = cls.translation(0, p2)
        return m3 @ m2 @ m1
    
    @classmethod
    def rotation(cls, p1, a1):
        if p1 == 0:
            return MobiusTransform([[np.exp(1j*a1), 0], [0, 1]])
        return cls.translation(p1, 0) @ cls.rotation(0, a1) @ cls.translation(0, p1)
    
    @classmethod
    def center_of_mass(cls, points, masses=None):
        return poincare_centroid(points, masses)

    @classmethod
    def distance_to_origin(cls, p):
        return 2 * np.arctanh(np.linalg.norm(p))

    @classmethod
    def angle_to_axis(cls, p):
        return np.arctan2(p.imag, p.real)

    @classmethod
    def point_along_axis(cls, x):
        return np.sign(x) * np.tanh(np.abs(x) / 2)


In [ ]:
def _rot_x_mat(a1):
    return np.array([
        [1, 0,           0         ],
        [0, np.cos(a1), -np.sin(a1)],
        [0, np.sin(a1),  np.cos(a1)]
    ])

def _rot_z_mat(a1):
    return np.array([
        [ np.cos(a1), np.sin(a1), 0,],
        [-np.sin(a1), np.cos(a1), 0], 
        [ 0         , 0         , 1]
    ])

class UnitSphereModel(Geometry):
    @classmethod
    def origin(cls):
        return np.array([1, 0, 0])

    @classmethod
    def translation(cls, p1, p2):
        def origin_translation_mat(p1):
            a1 = np.arctan2(p1[2], p1[1])
            m1 = _rot_x_mat(-a1)
            m2 = _rot_z_mat(-np.arccos(p1[0]))
            m3 = _rot_x_mat(a1)
            return m3 @ m2 @ m1
        def minus(p):
            return np.array([p[0], *-p[1:]])
        m1 = origin_translation_mat(minus(p1))
        m2 = origin_translation_mat(minus(m1 @ p2))
        m3 = origin_translation_mat(p1)
        
        mat = m3 @ m2 @ m1
        
        def translate(p):
            return p @ mat
        
        return translate

    @classmethod
    def _rotate_around_origin(cls, a1):
        mat = _rot_x_mat(a1)
        
        def origin_rotate(p):
            return p @ mat
        
        return origin_rotate

    @classmethod
    def center_of_mass(cls, points, masses=None):
        if masses is not None:
            masses = masses / np.sum(masses) * len(points)
            points = points * masses[..., None]
        result = np.mean(points, axis=0)
        result /= np.linalg.norm(result)
        return result

    @classmethod
    def distance_to_origin(cls, p):
        return np.arccos(np.clip(p[0], -1, 1))

    @classmethod
    def angle_to_axis(cls, p):
        return np.arctan2(p[2], p[1])

    @classmethod
    def point_along_axis(cls, x):
        return np.array([np.cos(x), np.sin(x), 0])

In [ ]:
from matplotlib import pyplot as plt
eps = 1e-6

for geo in (UnitSphereModel, EuclideanGeometry, PoincareDiskModel):
    print(f'testing {geo.__name__}')
    
    a = geo.origin()
    b = geo.from_polar(1, 0)
    c = geo.from_polar(2, np.pi/4)
    
    assert abs(geo.distance(c, geo.origin()) - 2) < eps
    assert np.linalg.norm(geo.translation(geo.origin(), c)(geo.origin()) - c) < eps, f'{geo.translation(geo.origin(), c)(geo.origin()), c}'
    assert geo.distance(geo.translation(b, c)(b), c) < eps, f'{geo.translation(b, c)(b), c}'
    
    rot_center = geo.from_polar(4, np.random.rand())
    a1 = np.random.rand()
    r = geo.rotation(rot_center, a1)
    r_inv = geo.rotation(rot_center, -a1)
    
    assert abs(geo.angle(c, a, b) - geo.angle(r(c), r(a), r(b))) < eps, f'{geo.angle(c, a, b), geo.angle(r(c), r(a), r(b))}'

    masses = np.random.rand(3)
    a1 = np.random.rand()

    assert geo.distance(r(r_inv(b)), b) < eps
    assert abs(geo.distance(a, c) - geo.distance(r(a), r(c))) < eps
    
    c1 = r(geo.center_of_mass(np.array([a, b, c]), masses))
    c2 = geo.center_of_mass(np.array([r(a), r(b), r(c)]), masses)
    assert geo.distance(c1, c2) < eps, f'{c1}, {c2}'

In [ ]:
r([0,1])

In [ ]:
np.arccos(1)

In [ ]:
2 * np.arctanh(0.9)

In [ ]:
np.random.rand(3)